# COMMU Latent UMAP Explorer

This notebook is designed for two goals:

1. **Study mode**: inspect latent geometry and per-segment losses interactively.
2. **Conference mode**: generate clean, exportable figures from the same analysis.

The notebook is intentionally explicit about what is already stored in the current COMMU cache and what still requires recomputation.

## Scope and honesty

The current COMMU cache is good enough for a useful presentation, but we should label the losses carefully.

What is already available in `COMMUDataset/losses/`:

- `z_chd`
- `z_txt`
- `final_loss`
- `kl_loss`, `kl_chd`, `kl_rhy`
- COMMU metadata such as `track_role`, `genre`, `inst`, `sample_rhythm`, and `time_signature`

For the argument you want to make, the main didactic losses are better framed as:

- **Total loss**: overall optimization view
- **Reconstruction loss**: global reconstruction-quality view
- **Chord loss / chroma loss**: harmony-oriented views
- **Duration loss**: texture / rhythmic proxy

The notebook will use those richer losses automatically **if enriched COMMU files are available**. If they are not available yet, the notebook still works and keeps the KL views as secondary study signals.


In [ ]:
from pathlib import Path
import inspect
import sys

REPO_ROOT = Path('/workspace/vae-textures-dev')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / 'base_model') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'base_model'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import Markdown, display

from commu_umap_support import (
    default_paths,
    load_commu_loss_table,
    compute_umap_embedding,
    metric_availability_table,
)
from base_model.model import DisentangleVAE


In [ ]:
REPO_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'NotebooksVAESymTex' else Path('/workspace/vae-textures-dev')
PATHS = default_paths(REPO_ROOT)
LOSS_DIR = PATHS.loss_dir
CACHE_PATH = PATHS.cache_table

# Use MAX_FILES for a quick rehearsal. Keep None for the full COMMU table.
MAX_FILES = None

# You can reduce these if you want faster exploratory runs.
DEFAULT_N_NEIGHBORS = 20
DEFAULT_MIN_DIST = 0.08
DEFAULT_METRIC = 'cosine'
DEFAULT_RANDOM_STATE = 42

print('Repo root:', REPO_ROOT)
print('Loss dir:', LOSS_DIR)
print('Cache path:', CACHE_PATH)

## UMAP configuration

This notebook uses UMAP separately for each selected latent space.

- `z_chd`: harmony-oriented latent only
- `z_txt`: texture-oriented latent only
- `z_both`: concatenation of `z_chd` and `z_txt` into one joint vector

So the default analysis does **not** mix `z_chd` into `z_txt` unless you explicitly choose `z_both`.

The explicit UMAP defaults are:

- `n_neighbors = 20`
- `min_dist = 0.08`
- `metric = 'cosine'`
- `random_state = 42`

These values are chosen for a stable and reproducible view, but they are meant to stay easy to edit for discussions with your supervisor.


In [ ]:
umap_settings = {
    'n_neighbors': DEFAULT_N_NEIGHBORS,
    'min_dist': DEFAULT_MIN_DIST,
    'metric': DEFAULT_METRIC,
    'random_state': DEFAULT_RANDOM_STATE,
}
umap_settings


## Source visibility

The helper functions are printed below on purpose so the notebook stays transparent about what is being measured.

In [ ]:
print(inspect.getsource(load_commu_loss_table))
print(inspect.getsource(compute_umap_embedding))
print(inspect.getsource(metric_availability_table))
print(inspect.getsource(DisentangleVAE.loss_function))

## Available metrics in the current COMMU cache

In [ ]:
availability = metric_availability_table(table if 'table' in globals() else None)
availability


## Load the COMMU segment table

On the first full run, this may take a while because it scans all `COMMUDataset/losses/*.npz` files. After that, it should reuse the pickle cache.

If enriched per-segment losses exist in `COMMUDataset/losses_enriched/`, they are merged in automatically.


In [ ]:
table = load_commu_loss_table(
    LOSS_DIR,
    enriched_loss_dir=PATHS.enriched_loss_dir,
    max_files=MAX_FILES,
    use_cache=True,
    cache_path=CACHE_PATH,
)

print(f'Rows loaded: {len(table):,}')
display(table[['segment_id', 'track_role', 'track_role_grouped', 'genre', 'inst', 'final_loss', 'kl_chd', 'kl_rhy']].head())
display(metric_availability_table(table))
print(table['track_role'].value_counts(dropna=False).to_string())


## Embedding cache

This notebook caches the computed UMAP coordinates in memory so you can switch between colorings without recomputing the same embedding.

In [ ]:
embedding_cache = {}

def get_embedding(latent_space: str) -> pd.DataFrame:
    if latent_space not in embedding_cache:
        embedding_cache[latent_space] = compute_umap_embedding(
            table,
            latent_space=latent_space,
            n_neighbors=DEFAULT_N_NEIGHBORS,
            min_dist=DEFAULT_MIN_DIST,
            metric=DEFAULT_METRIC,
            random_state=DEFAULT_RANDOM_STATE,
        )
    return embedding_cache[latent_space].copy()

## Interactive explorer

How to use it:

- choose the latent space (`z_chd`, `z_txt`, or both concatenated)
- choose the coloring
- click a point to inspect all metadata and all currently available losses
- use the Plotly camera button to download a figure, or just screenshot it

Recommended conference views:

- `track_role_detailed` or `track_role_grouped` for the melody split story
- `final_loss` for the overall optimization story
- `chord_loss` or `chroma_loss` for a harmony-oriented story, when enriched losses exist
- `duration_loss` for a texture/rhythm-oriented story, when enriched losses exist

Two melody views are included:

- `track_role_detailed`: keeps `main_melody` and `sub_melody` separate
- `track_role_grouped`: merges both into `melody`


In [ ]:
COLOR_LABELS = {
    'Detailed melody role': 'track_role_detailed',
    'Grouped melody role': 'track_role_grouped',
    'Track role (raw)': 'track_role',
    'Genre': 'genre',
    'Instrument': 'inst',
    'Sample rhythm': 'sample_rhythm',
    'Time signature': 'time_signature',
    'Pitch range': 'pitch_range',
    'Total loss': 'final_loss',
    'Reconstruction loss': 'recon_loss',
    'Harmony proxy: chord loss': 'chord_loss',
    'Harmony detail: chroma loss': 'chroma_loss',
    'Texture proxy: duration loss': 'duration_loss',
    'Pitch loss': 'pitch_loss',
    'Chord KL': 'kl_chd',
    'Texture KL': 'kl_rhy',
    'Overall KL': 'kl_loss',
    'Tempo (BPM)': 'bpm',
}

AVAILABLE_COLOR_LABELS = {label: column for label, column in COLOR_LABELS.items() if column in table.columns and table[column].notna().any()}
NUMERIC_FIELDS = {'final_loss', 'recon_loss', 'chord_loss', 'chroma_loss', 'duration_loss', 'pitch_loss', 'kl_chd', 'kl_rhy', 'kl_loss', 'bpm', 'num_measures'}
DETAIL_COLUMNS = [
    'segment_id', 'track_id', 'segment_index', 'track_role_detailed', 'track_role_grouped',
    'genre', 'inst', 'sample_rhythm', 'time_signature', 'pitch_range', 'audio_key',
    'bpm', 'num_measures', 'final_loss', 'recon_loss', 'chord_loss', 'chroma_loss', 'duration_loss',
    'pitch_loss', 'kl_loss', 'kl_chd', 'kl_rhy', 'chord_progressions',
]

latent_space_dropdown = widgets.Dropdown(
    options=[('Chord latent z_chd', 'z_chd'), ('Texture latent z_txt', 'z_txt'), ('Concatenated z_chd + z_txt', 'z_both')],
    value='z_txt',
    description='Latent:',
    layout=widgets.Layout(width='340px')
)

default_color = 'track_role_detailed' if 'track_role_detailed' in AVAILABLE_COLOR_LABELS.values() else next(iter(AVAILABLE_COLOR_LABELS.values()))
color_field_dropdown = widgets.Dropdown(
    options=list(AVAILABLE_COLOR_LABELS.items()),
    value=default_color,
    description='Color by:',
    layout=widgets.Layout(width='440px')
)

marker_size_slider = widgets.IntSlider(value=6, min=3, max=12, step=1, description='Marker size:', continuous_update=False)
opacity_slider = widgets.FloatSlider(value=0.78, min=0.2, max=1.0, step=0.02, description='Opacity:', continuous_update=False)

plot_output = widgets.Output()
detail_output = widgets.HTML(value='<b>Click a point to inspect the selected segment.</b>')
last_figure = {'value': None}

def _format_value(value):
    if isinstance(value, float):
        return f'{value:.4f}'
    text = str(value)
    return text if len(text) < 220 else text[:220] + ' ...'

def _detail_html(row: pd.Series) -> str:
    lines = ['<h4>Selected segment</h4>', '<table>']
    for column in DETAIL_COLUMNS:
        if column in row.index and column in table.columns:
            lines.append(f"<tr><td style='padding-right:12px'><b>{column}</b></td><td>{_format_value(row[column])}</td></tr>")
    lines.append('</table>')
    return ''.join(lines)

def build_figure(df: pd.DataFrame, color_field: str, marker_size: int, opacity: float):
    hover_columns = [
        'segment_id', 'track_role_detailed', 'track_role_grouped', 'genre', 'inst',
        'sample_rhythm', 'time_signature', 'pitch_range', 'audio_key',
        'final_loss', 'recon_loss', 'chord_loss', 'chroma_loss', 'duration_loss', 'pitch_loss',
        'kl_chd', 'kl_rhy', 'kl_loss', 'bpm'
    ]
    hover_columns = {column: True for column in hover_columns if column in df.columns}

    fig = px.scatter(
        df,
        x='umap_x',
        y='umap_y',
        color=color_field,
        hover_name='segment_id',
        hover_data=hover_columns,
        custom_data=['segment_id'],
        render_mode='webgl',
        template='plotly_white',
        title=f'COMMU UMAP | latent={latent_space_dropdown.value} | color={color_field}',
        width=1180,
        height=760,
        color_continuous_scale='Viridis' if color_field in NUMERIC_FIELDS else None,
    )
    fig.update_traces(marker=dict(size=marker_size, opacity=opacity, line=dict(width=0)))
    fig.update_layout(legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0), margin=dict(l=20, r=20, t=80, b=20), title_font_size=22)
    return go.FigureWidget(fig)

def render_interactive_plot(*_):
    df = get_embedding(latent_space_dropdown.value)
    color_field = color_field_dropdown.value
    with plot_output:
        plot_output.clear_output(wait=True)
        fig = build_figure(df, color_field, marker_size_slider.value, opacity_slider.value)

        def handle_click(trace, points, state):
            if not points.point_inds:
                return
            point_idx = points.point_inds[0]
            segment_id = trace.customdata[point_idx][0]
            row = df.loc[df['segment_id'] == segment_id].iloc[0]
            detail_output.value = _detail_html(row)

        for trace in fig.data:
            trace.on_click(handle_click)

        last_figure['value'] = fig
        display(fig)

for widget in [latent_space_dropdown, color_field_dropdown, marker_size_slider, opacity_slider]:
    widget.observe(render_interactive_plot, names='value')

controls = widgets.HBox([latent_space_dropdown, color_field_dropdown])
style_controls = widgets.HBox([marker_size_slider, opacity_slider])
display(controls)
display(style_controls)
display(widgets.HBox([plot_output, detail_output]))
render_interactive_plot()


## Conference-friendly export

The interactive figure already has Plotly's built-in camera button.

If you want a dedicated export-ready figure for a specific view, use the cell below to regenerate a clean static-style Plotly figure and then either:

- click the camera button
- save as HTML
- or screenshot it at a large browser zoom level

In [ ]:
EXPORT_LATENT = 'z_txt'
EXPORT_COLOR = 'track_role_detailed' if 'track_role_detailed' in table.columns else 'track_role'

export_df = get_embedding(EXPORT_LATENT)
export_fig = px.scatter(
    export_df,
    x='umap_x',
    y='umap_y',
    color=EXPORT_COLOR,
    hover_name='segment_id',
    render_mode='webgl',
    template='plotly_white',
    title=f'COMMU UMAP export view | latent={EXPORT_LATENT} | color={EXPORT_COLOR}',
    width=1400,
    height=900,
)
export_fig.update_traces(marker=dict(size=7, opacity=0.82, line=dict(width=0)))
export_fig.update_layout(title_font_size=26, legend=dict(orientation='h', yanchor='bottom', y=1.01, x=0))
export_fig.show()

# Optional HTML export:
# export_fig.write_html('commu_umap_export.html')


## Next step for exact harmony / texture losses

When you want the notebook to color by a fuller decomposition, use the support script below to recompute richer per-segment files that explicitly store:

- `recon_loss`
- `chord_loss`
- `root_loss`, `chroma_loss`, `bass_loss`
- `pitch_loss`, `duration_loss`

That script is already in this folder as `generate_commu_enriched_loss_dataset.py`.

In [ ]:
script_path = Path('generate_commu_enriched_loss_dataset.py')
print(script_path.read_text()[:5000])